In [ ]:
%%capture
# Skip restarting message in Colab
import sys; modules = list(sys.modules.keys())
for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None

# 1. Install vLLM first (0.19.1 is a solid choice for Colab)
!pip install vllm==0.19.1

# 2. Install Unsloth with its specific Colab dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 3. Install the "Big Three" for GRPO with strict versions
# Note: We put trl==0.14.0 here to ensure it isn't overwritten
!pip install --no-deps trl==0.14.0 peft accelerate bitsandbytes xformers

# 4. Install supporting utilities
!pip install --no-deps unsloth_zoo torchcodec
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps diffusers

# 5. Fix Transformers to a version compatible with Unsloth's current hooks
!pip install --upgrade "transformers>=4.48.0" 
!pip install --upgrade pillow

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    # Gemma 4 requires transformers >= 5.5.0 — do NOT pin to 4.x here
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
# Gemma 4 requires transformers >= 5.5.0
!uv pip install --upgrade --no-deps "transformers>=5.5.0" "tokenizers>=0.22.0,<=0.23.0" "trl>=0.28.0" unsloth unsloth_zoo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
save_path = "/content/drive/MyDrive/UnslothGRPO/tictactoeExample"
# Restart Jyputer

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

In [ ]:
#from unsloth import is_bfloat16_supported
#from datasets import Dataset
#import torch
#import pandas as pd

# NOTES

It is playing tic-tac-toe, it gets to the point where it loses. It does not draw or win, also it is playing against a perfect player. Ofcourse there are many possible states, which is hard for the llm.

- are the rewards correct?
- are the completion_ids correct?


Why is the "game_reward" half the "reward"? Because the reward function adds a 0.

Is tokenization done correctly in multi_turn_generation? Like padding?

It often chooses an index (move/action) that the opponent already has played. Does it get the game state correctly? If it doesn't, then it makes a lot of sense as why it doesn't learn to win. Because then it is guessing.

# Model Loading

In [ ]:
from unsloth import FastModel
import torch

temperature = 0.8
max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

#model_name = "unsloth/gemma-4-e4b-it-bnb-4bit"
model_name = "unsloth/gemma-4-E2B-it"

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    dtype = None, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

from vllm import SamplingParams
import random
import io
from collections import Counter

sampling_params = SamplingParams(
    temperature = temperature,
    top_p = 0.95,
    max_tokens = 512,
)

In [ ]:
#model = FastLanguageModel.get_peft_model(
#    model,
#    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
#    target_modules = [
#        "q_proj", "k_proj", "v_proj", "o_proj",
#        "gate_proj", "up_proj", "down_proj",
#    ], # Remove QKVO if out of memory
#    lora_alpha = lora_rank,
#    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
#    random_state = 3407,
#)


In [ ]:
#max_seq_length = 2048 # Can increase for longer reasoning traces
#lora_rank = 32 # Larger rank = smarter, but slower

#model_name = "unsloth/gemma-4-e4b-it-bnb-4bit"

#model, tokenizer = FastLanguageModel.from_pretrained(
#    model_name = model_name,
#    max_seq_length = max_seq_length,
#    load_in_4bit = True, # False for LoRA 16bit
#    fast_inference = True, # Enable vLLM fast inference
#    max_lora_rank = lora_rank,
#    gpu_memory_utilization = 0.5, # Reduce if out of memory
#)

#model = FastLanguageModel.get_peft_model(
#    model,
#    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
#    target_modules = [
#        "q_proj", "k_proj", "v_proj", "o_proj",
#        "gate_proj", "up_proj", "down_proj",
#    ], # Remove QKVO if out of memory
#    lora_alpha = lora_rank,
#    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
#    random_state = 3407,
#)

#from vllm import SamplingParams
#import random
#import io
#from collections import Counter

#temperature = 0.8

#sampling_params = SamplingParams(
#    temperature = temperature,
#    top_p = 0.95,
#    max_tokens = 512,
#)

# Tic-Tac-Toe Game

In [ ]:
import math

class TicTacToeGame:
    def __init__(self):
        # The board is a list of 9 spaces. Positions are numbered 0 to 8.
        self.board = [' ']*9
        self.current_winner = None  # Track the winner!
        self.player_letter = 'X'    # Human is X
        self.ai_letter = 'O'        # AI is O
        self.done = False           # Game over flag

    def print_board(self):
        # Prints the board in a 3x3 grid format.
        for row in [self.board[i*3:(i+1)*3] for i in range(3)]:
            print("| " + " | ".join(row) + " |")
        print()

    def get_state(self):
        # Return a copy of the current board state.
        board = self.board.copy()
        board_string = ', '.join(f"{i}: {'(empty)' if board[i] == ' ' else board[i]}" for i in range(9))

        return board_string

    def available_moves(self):
        # Returns a list of indices for available moves.
        return [i for i, spot in enumerate(self.board) if spot == ' ']

    def move(self, square, letter):
        # If the move is valid (square is empty), assign the letter to that square.
        if self.board[square] == ' ':
            self.board[square] = letter
            if self.winner(square, letter):
                self.current_winner = letter
            return True
        return False

    def winner(self, square, letter):
        # Check the row, column, and diagonals for a win.
        row_ind = square // 3
        row = self.board[row_ind*3:(row_ind+1)*3]
        if all(spot == letter for spot in row):
            return True

        col_ind = square % 3
        column = [self.board[col_ind+i*3] for i in range(3)]
        if all(spot == letter for spot in column):
            return True

        # Check diagonals (only if the square is even)
        if square % 2 == 0:
            diagonal1 = [self.board[i] for i in [0, 4, 8]]
            if all(spot == letter for spot in diagonal1):
                return True
            diagonal2 = [self.board[i] for i in [2, 4, 6]]
            if all(spot == letter for spot in diagonal2):
                return True

        return False

    def is_draw(self):
        # The game is a draw if there are no empty spaces and no winner.
        return ' ' not in self.board

    def get_reward(self):
        """
        Returns the reward from the human player's perspective:
          +1 if the player wins,
          -1 if the AI wins,
           0 for a draw or if the game is still ongoing.
        """
        if self.current_winner == self.player_letter:
            return 3
        elif self.current_winner == self.ai_letter:
            return 0
        elif self.is_draw():
            return 1
        else:
            return 0

    def player_move(self, square):
        """
        Performs the player's move at the given square.
        Returns a tuple: (current_state, reward).
        If the move ends the game (win/draw), it sets the done flag.
        Otherwise, after the player’s move, the AI makes its move.
        """
        if self.done:
            print("Game is already over!")
            return self.get_state(), self.get_reward()

        if self.move(square, self.player_letter):
            if self.current_winner or self.is_draw():
                self.done = True
                reward = self.get_reward()
                return self.get_state(), reward
            else:
                # Let the AI make its move.
                self.ai_move()
                if self.current_winner or self.is_draw():
                    self.done = True
                reward = self.get_reward()
                return self.get_state(), reward
        else:
            return "Invalid move!", 0

    def ai_move(self):
        # Use minimax to pick the best move for the AI.
        best_move = self.minimax(self.board, self.ai_letter)['position']
        self.move(best_move, self.ai_letter)

    def minimax(self, board, player):
        """
        A recursive minimax algorithm.
        Returns a dictionary with the best 'position' and 'score' for the given player.
        The AI (self.ai_letter) is the maximizer.
        """
        max_player = self.ai_letter  # AI is the maximizing player
        other_player = self.player_letter if player == self.ai_letter else self.ai_letter

        avail = [i for i, spot in enumerate(board) if spot == ' ']

        # Base case: check if the previous move (by other_player) is a winner.
        if self.check_winner(board, other_player):
            return {
                'position': None,
                'score': (len(avail) + 1) if other_player == max_player else - (len(avail) + 1)
            }
        elif not avail:  # No more moves = draw.
            return {'position': None, 'score': 0}

        # Initialize the best move dictionary.
        if player == max_player:
            best = {'position': None, 'score': -math.inf}  # maximize score
        else:
            best = {'position': None, 'score': math.inf}   # minimize score

        for possible_move in avail:
            board[possible_move] = player
            # Check if this move wins the game immediately.
            if self.check_winner(board, player):
                score = (len([s for s in board if s == ' ']) + 1) if player == max_player else - (len([s for s in board if s == ' ']) + 1)
                result = {'position': possible_move, 'score': score}
            else:
                # Recurse: simulate the game after this move.
                result = self.minimax(board, other_player)
                result['position'] = possible_move
            board[possible_move] = ' '  # Undo move

            if player == max_player:
                if result['score'] > best['score']:
                    best = result
            else:
                if result['score'] < best['score']:
                    best = result

        return best

    def check_winner(self, board, letter):
        # A helper function that checks if the given letter has won on the provided board.
        # Check rows.
        for row in range(3):
            if board[row*3] == board[row*3+1] == board[row*3+2] == letter:
                return True
        # Check columns.
        for col in range(3):
            if board[col] == board[col+3] == board[col+6] == letter:
                return True
        # Check diagonals.
        if board[0] == board[4] == board[8] == letter:
            return True
        if board[2] == board[4] == board[6] == letter:
            return True
        return False



# System Prompt & Format Reward Functions

In [ ]:
SYSTEM_PROMPT = """
You are playing a game of tic-tac-toe, it is your turn. You are X and the opponent is O, give your action with a number (0-8).

Reason in the following format:
<reasoning>
...
</reasoning>
Give your final action with a number (0-8) in the following format:
<action>
...
</action>
"""


def game_reward(completions, **kwargs) -> list[float]:
    # Dummy function to show the results in the log
    # Values will be filled the the CustomGRPOTrainer, directly extracted from the interactive gameplay.
    return [0.0 for completion in completions]

# Game Reward & Prompt Creation

In [ ]:
def create_database():
    """
    Usually you have a dataset with many questions/prompts, with games there is only the begin state.
    So that is all we need? Multiplied by the number of games. Does that even make sense? Maybe we only
    need one begin state and then set the number of generations very high.
    """
    game_state = TicTacToeGame().get_state()

    """
    prompt = tokenizer.apply_chat_template([
            {"role" : "system", "content" : SYSTEM_PROMPT},
            {"role" : "user", "content" : f"Game state: {game_state}"},
        ], tokenize = False, add_generation_prompt = True)
    """

    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Game state: {game_state}"}
    ]
    snapshots = []
    snapshots.append({
        "prompt": prompt,
        "answer": "", #placeholder, we don't have a ground truth
    })

    return snapshots

# Tic-Tac-Toe Main Test

In [ ]:
# --- Main game loop ---
"""
if __name__ == '__main__':
    game = TicTacToeGame()
    print("Welcome to Tic Tac Toe!")
    print("You are 'X' and the AI is 'O'.")
    print("The board positions are numbered 0 to 8 as follows:")
    print("| 0 | 1 | 2 |")
    print("| 3 | 4 | 5 |")
    print("| 6 | 7 | 8 |\n")

    game.print_board()

    # Main loop: continue until the game is done.
    while not game.done:
        valid_move = False
        while not valid_move:
            try:
                move = int(input("Enter your move (0-8): "))
                if move in game.available_moves():
                    valid_move = True
                else:
                    print("That square is not available. Try again.")
            except ValueError:
                print("Please enter a valid number between 0 and 8.")

        # Player makes a move and receives (state, reward)
        state, reward = game.player_move(move)
        game.print_board()
        print("Current Board State:", state)
        print("Reward:", reward, "\n")

        # If the game is over, announce the result.
        if game.done:
            if game.current_winner == game.player_letter:
                print("Congratulations! You win!")
            elif game.current_winner == game.ai_letter:
                print("You lose! The AI wins.")
            else:
                print("It's a draw!")
            print("Final Reward:", reward)
"""

# GRPO Config

In [ ]:
import builtins
# This forces 'top_k' to exist everywhere, satisfying the broken script
builtins.top_k = -1

from unsloth import is_bfloat16_supported
from trl import GRPOConfig, GRPOTrainer
import re
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    temperature = temperature,
    # Fix: Explicitly pass sampling params to satisfy the Unsloth patcher
    vllm_sampling_params = {
        "top_k": -1,
        "top_p": 0.95,
    },
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.01,
    #beta=2,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_seq_length-512,
    max_completion_length = 512,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 200,
    save_steps = 50,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = f"{save_path}/outputs",
)

# CustomGRPOTrainer

In [ ]:
from collections import defaultdict
import os
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import textwrap
import warnings
import re

from accelerate.utils import broadcast_object_list, gather, gather_object, set_seed
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from trl import maybe_apply_chat_template, apply_chat_template
from trl.data_utils import is_conversational
from trl import maybe_apply_chat_template
from trl.trainer.grpo_trainer import Any, Union, pad

# IMPORTANT: Import the patched Unsloth trainer.
#from unslothgrpotrainer import UnslothGRPOTrainer
from unsloth_compiled_cache.UnslothGRPOTrainer import UnslothGRPOTrainer

# Define your external action function.
def perform_external_action(action='', game=None):
    # Replace this stub with your actual external action logic.
    try:
        move = int(action.strip())
        state, reward = game.player_move(move)
        return state, reward
    except:
        return "Invalid move!", 0

class CustomGRPOTrainer(UnslothGRPOTrainer):
    def __init__(self, *args, game_object=None, sampling_params=None, **kwargs):
        self.game_object = game_object
        self.sampling_params = sampling_params
        super().__init__(*args, **kwargs)
        self.num_iterations = 1
        self._metrics = defaultdict(list)


    def multi_turn_generation(self, prompt, model, tokenizer, generation_config, max_new_tokens=50, game_object=None):
        """
        Generate responses and handle the multi-turn  interaction.
        Returns the complete sequence of token IDs for the game and the accumulated reward.

        REWARDS:
        Win: +3 (never happens)
        Draw: +1
        Loss: +0
        Per wrong action format: -1
        Correct action format: +0.1
        Per valid action: +0.1
        Invalid move: +0 (penalize harder) -2
        No action tag: +0
        """

        print("############## New Game ###########################################################")
        # Initialize a new game instance
        game = self.game_object() if self.game_object else None
        if game is None:
            raise ValueError("Game object factory not provided")

        device = next(model.parameters()).device
        full_text = prompt
        total_reward = 0
        completion_ids = []

        # Pre-load Lora
        lora_request = model.load_lora('grpo_trainer_lora_model', load_tensors=True)
        lora_request = None
        while not game.done:
            outputs = self.llm.generate(
                [full_text],
                sampling_params=self.sampling_params,
                use_tqdm=False,
                lora_request=lora_request
            )

            # Get the generated text and tokens
            new_text = outputs[0].outputs[0].text
            new_token_ids = outputs[0].outputs[0].token_ids

            # Add to our tracking variables
            completion_ids.extend(new_token_ids)
            full_text += new_text

            # Check if the model provided a complete action
            action_match = re.search(r'<action>\s*([0-8])\s*</action>', new_text, re.DOTALL)

            # Check for the stop token
            if action_match:
                # small reward for the right format
                total_reward += 0.1

                # Extract the action
                action = action_match.group(1).strip()

                print(f"[Extracted action: {action}]")
                game_state, reward = perform_external_action(action, game)
                total_reward += reward
                print(f"[External action executed. Result: {game_state}, Game Reward: {reward}, Total Reward: {total_reward}]")

                if game_state == "Invalid move!":
                    total_reward -= 2
                    print("Invalid move!")
                    break

                # small reward for a valid move.
                total_reward += 0.1

                if game.done:
                    # let's break it before feedback is added to the full_text, or should we?
                    print(f"Game is over! Winner: {game.current_winner}")
                    break

                feedback = self.processing_class.apply_chat_template(
                    [{"role": "system", "content": ""},
                     {"role": "user", "content": f"Game state: {game_state}"}],
                    add_generation_prompt=True,
                    tokenize=False
                )
                full_text += feedback
                #completion_ids.extend(tokenizer(feedback)["input_ids"])
            else:
                print("No <action> digit </action> found; finishing generation.")
                total_reward -= 1
                break

        #print(f"Full Output Text: {full_text}")
        return completion_ids, total_reward

    def _prepare_inputs(self, inputs: dict[str, Union[torch.Tensor, Any]]) -> dict[str, Union[torch.Tensor, Any]]:
        # Ensure we have a generation configuration. If using vLLM, use sampling_params.
        if not hasattr(self, "generation_config"):
            from transformers import GenerationConfig
            if self.args.use_vllm:
                self.generation_config = self.sampling_params  # Use sampling_params for vLLM
            else:
                self.generation_config = GenerationConfig(
                    max_new_tokens=self.max_completion_length,
                    do_sample=True,
                    temperature=self.args.temperature,
                    pad_token_id=self.processing_class.pad_token_id,
                )

        device = self.accelerator.device
        # Prepare prompts and apply the chat template.

        prompts = [x["prompt"] for x in inputs]
        prompts_text = [
            maybe_apply_chat_template(example, self.processing_class)["prompt"]
            for example in inputs
        ]
        # Tokenize prompts.
        prompt_inputs = self.processing_class(
            prompts_text, return_tensors="pt", padding=True, padding_side="left", add_special_tokens=False
        )
        prompt_ids, prompt_mask = prompt_inputs["input_ids"], prompt_inputs["attention_mask"]

        prompt_ids = prompt_ids.to(device)
        prompt_mask = prompt_mask.to(device)

        if self.max_prompt_length is not None:
            prompt_ids = prompt_ids[:, -self.max_prompt_length:]
            prompt_mask = prompt_mask[:, -self.max_prompt_length:]

        if self.args.use_vllm:
            # First, have main process load weights if needed
            if self.state.global_step != self._last_loaded_step:
                self._move_model_to_vllm()
                self._last_loaded_step = self.state.global_step

            # Generate completions using vLLM: gather all prompts and use them in a single call in the main process
            all_prompts_text = gather_object(prompts_text)

            if self.accelerator.is_main_process:
                # Use custom generation for each prompt.
                completion_ids = []
                game_rewards = []
                for prompt in all_prompts_text:
                    # llm.generate(all_prompts_text)
                    single_completion_ids, reward = self.multi_turn_generation(
                        prompt, self.model, self.processing_class, self.generation_config,
                        max_new_tokens=self.max_completion_length,
                    )
                    completion_ids.append(single_completion_ids)
                    game_rewards.append(reward)
                #completion_ids = [out.token_ids for completions in outputs for out in completions.outputs]
            else:
                completion_ids = [None] * len(all_prompts_text)

            # Broadcast results to all processes
            completion_ids = broadcast_object_list(completion_ids, from_process=0)
            game_rewards = broadcast_object_list(game_rewards, from_process=0)

            # Convert rewards to tensor
            game_rewards_tensor = torch.tensor(
                game_rewards,
                dtype=torch.float32,
                device=device
            )

            # Get process-specific slice
            process_slice = slice(
                self.accelerator.process_index * len(prompts),
                (self.accelerator.process_index + 1) * len(prompts),
            )
            completion_ids = completion_ids[process_slice]

            # Pad the completions, and concatenate them with the prompts
            completion_ids = [torch.tensor(ids, device=device) for ids in completion_ids]
            completion_ids = pad(completion_ids, padding_value=self.processing_class.pad_token_id)
            prompt_completion_ids = torch.cat([prompt_ids, completion_ids], dim=1)
        else:
            #TODO: make this work
            # Regular generation path
            with unwrap_model_for_generation(self.model, self.accelerator) as unwrapped_model:
                prompt_completion_ids = unwrapped_model.generate(
                    prompt_ids, attention_mask=prompt_mask, generation_config=self.generation_config
                )

            # Compute prompt length and extract completion ids
            prompt_length = prompt_ids.size(1)
            prompt_ids = prompt_completion_ids[:, :prompt_length]
            completion_ids = prompt_completion_ids[:, prompt_length:]

        # Mask everything after the first EOS token
        is_eos = completion_ids == self.processing_class.eos_token_id
        eos_idx = torch.full((is_eos.size(0),), is_eos.size(1), dtype=torch.long, device=device)
        eos_idx[is_eos.any(dim=1)] = is_eos.int().argmax(dim=1)[is_eos.any(dim=1)]
        sequence_indices = torch.arange(is_eos.size(1), device=device).expand(is_eos.size(0), -1)
        completion_mask = (sequence_indices <= eos_idx.unsqueeze(1)).int()

        # Concatenate prompt_mask with completion_mask for logit computation
        attention_mask = torch.cat([prompt_mask, completion_mask], dim=1)  # (B*G, P+C)

        logits_to_keep = completion_ids.size(1)  # we only need to compute the logits for the completion tokens

        with torch.inference_mode(), torch.amp.autocast(device_type = 'cuda', dtype = torch.float16 if os.environ.get('ACCELERATE_MIXED_PRECISION', 'fp16') == 'fp16' else torch.bfloat16) if not torch.is_autocast_enabled('cuda') else nullcontext():
            if self.ref_model is not None:
                ref_per_token_logps, _ = self._get_per_token_logps_and_entropies(
                    self.ref_model, prompt_completion_ids, attention_mask, logits_to_keep
                )
            else:
                with self.accelerator.unwrap_model(self.model, keep_fp32_wrapper = False).disable_adapter():
                    ref_per_token_logps, _ = self._get_per_token_logps_and_entropies(
                        self.model, prompt_completion_ids, attention_mask, logits_to_keep
                    )

        # Decode the generated completions
        completions_text = self.processing_class.batch_decode(completion_ids, skip_special_tokens=True)
        if is_conversational(inputs[0]):
            completions = []
            for prompt, completion in zip(prompts, completions_text):
                bootstrap = prompt.pop()["content"] if prompt[-1]["role"] == "assistant" else ""
                completions.append([{"role": "assistant", "content": bootstrap + completion}])
        else:
            completions = completions_text

        rewards_per_func = torch.zeros(len(prompts), len(self.reward_funcs), device=device)
        for i, (reward_func, reward_processing_class) in enumerate(
            zip(self.reward_funcs, self.reward_processing_classes)
        ):
            if isinstance(reward_func, nn.Module):  # Module instead of PretrainedModel for compat with compiled models
                if is_conversational(inputs[0]):
                    messages = [{"messages": p + c} for p, c in zip(prompts, completions)]
                    texts = [apply_chat_template(x, reward_processing_class)["text"] for x in messages]
                else:
                    texts = [p + c for p, c in zip(prompts, completions)]
                reward_inputs = reward_processing_class(
                    texts, return_tensors="pt", padding=True, padding_side="right", add_special_tokens=False
                )
                reward_inputs = super()._prepare_inputs(reward_inputs)
                with torch.inference_mode(), torch.amp.autocast(device_type = 'cuda', dtype = torch.float16 if os.environ.get('ACCELERATE_MIXED_PRECISION', 'fp16') == 'fp16' else torch.bfloat16) if not torch.is_autocast_enabled('cuda') else nullcontext():
                    rewards_per_func[:, i] = reward_func(**reward_inputs).logits[:, 0]  # Shape (B*G,)
            else:
                # Repeat all input columns (but "prompt" and "completion") to match the number of generations
                keys = [key for key in inputs[0] if key not in ["prompt", "completion"]]
                reward_kwargs = {key: [example[key] for example in inputs] for key in keys}
                output_reward_func = reward_func(prompts=prompts, completions=completions, **reward_kwargs)
                rewards_per_func[:, i] = torch.tensor(output_reward_func, dtype=torch.float32, device=device)

        # Gather the reward per function: this part is crucial, because the rewards are normalized per group and the
        # completions may be distributed across processes
        rewards_per_func = gather(rewards_per_func)# + game_rewards_tensor.unsqueeze(1)

        # Create a device-matched game rewards tensor and broadcast it
        game_rewards_tensor = game_rewards_tensor.to(device)
        game_rewards_tensor = gather(game_rewards_tensor)

        rewards_with_game = torch.cat([rewards_per_func, game_rewards_tensor.unsqueeze(1)], dim=1)

        # Create an extended weights tensor that includes the game reward weight
        # Setting game reward weight to 1.0 by default, but this could be configurable
        game_weight = torch.tensor([1.0], device=device)
        extended_weights = torch.cat([self.reward_weights.to(device), game_weight])

        # Apply weights to each reward function's output and sum
        rewards = (rewards_with_game * extended_weights.unsqueeze(0)).sum(dim=1)

        # Compute grouped-wise rewards
        mean_grouped_rewards = rewards.view(-1, self.num_generations).mean(dim=1)
        std_grouped_rewards = rewards.view(-1, self.num_generations).std(dim=1)

        # Normalize the rewards to compute the advantages
        mean_grouped_rewards = mean_grouped_rewards.repeat_interleave(self.num_generations, dim=0)
        std_grouped_rewards = std_grouped_rewards.repeat_interleave(self.num_generations, dim=0)
        advantages = (rewards - mean_grouped_rewards) / (std_grouped_rewards + 1e-4)

        # Slice to keep only the local part of the data
        process_slice = slice(
            self.accelerator.process_index * len(prompts),
            (self.accelerator.process_index + 1) * len(prompts),
        )
        advantages = advantages[process_slice]

        # Log the metrics
        reward_per_func = rewards_per_func.mean(0)
        for i, reward_func in enumerate(self.reward_funcs):
            if isinstance(reward_func, nn.Module):  # Module instead of PretrainedModel for compat with compiled models
                reward_func_name = reward_func.config._name_or_path.split("/")[-1]
            else:
                reward_func_name = reward_func.__name__
            self._metrics[f"rewards/{reward_func_name}"].append(reward_per_func[i].item())

        self._metrics["rewards/game_reward"].append(game_rewards_tensor.mean().item())
        self._metrics["reward"].append(rewards.mean().item())
        self._metrics["reward_std"].append(std_grouped_rewards.mean().item())

        if (
            self.log_completions
            and self.state.global_step % self.args.logging_steps == 0
            and "wandb" in self.args.report_to
        ):
            import pandas as pd
            import wandb
            # For logging
            table = {
                "step": [str(self.state.global_step)] * len(rewards),
                "prompt": gather_object(prompts_text),
                "completion": gather_object(completions_text),
                "reward": rewards.tolist(),
            }
            df = pd.DataFrame(table)

            if wandb.run is not None and self.accelerator.is_main_process:
                wandb.log({"completions": wandb.Table(dataframe=df)})

        return {
            "prompt_ids": prompt_ids,
            "prompt_mask": prompt_mask,
            "completion_ids": completion_ids,
            "completion_mask": completion_mask,
            "ref_per_token_logps": ref_per_token_logps,
            "advantages": advantages,
        }

# Train

In [ ]:
snapshot_df = pd.DataFrame(create_database())

train_dataset = Dataset.from_pandas(snapshot_df)

# Train the model for a few steps with the custom GRPO Trainer
trainer = CustomGRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[game_reward],
    args=training_args,
    train_dataset=train_dataset,
    game_object=TicTacToeGame,
    sampling_params=sampling_params,
)

trainer.train()

model.save_lora(f"{save_path}/grpo_saved_lora")

In [ ]:
from transformers import TrainerState

checkpoint_path = f"{save_path}/checkpoint"

# Save full checkpoint (including optimizer/scheduler)
trainer.save_model(checkpoint_path)
trainer.state.save_to_json(f"{checkpoint_path}/trainer_state.json")
model.save_lora(f"{save_path}/grpo_saved_lora")